# Prototype 19 — Real Fetch + Dev-DB Correlation

**Goal:** Run all URLs from `inputs/rd-urls2.txt` through the real three-parser cascade (`lib/rd_parsers.py`), then correlate each successfully parsed product against Aryan's real closet items in `~/.closet/closet.db`.

**Tradeoff acknowledged:** Pearson (Vertex) embedding is not wired. Instead, we use **TF-IDF** as a synthetic "hash-stub-embed" — deterministic text vectors derived from the parsed product fields and the closet items' metadata. This lets us measure **parser failure modes** and **closet-match relevance** without Vertex creds.

**Output:** Per-URL pass/fail table + top-5 closet matches per product with thumbnails.

**Requires:**
```bash
pip install httpx scikit-learn pillow matplotlib
```

In [1]:
import os, sys, json, time, sqlite3, re, hashlib
from pathlib import Path
from collections import Counter
from urllib.parse import urlparse

import httpx
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from dotenv import load_dotenv
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Path setup — walk up from cwd (or notebook file location) to find REPO_ROOT
_candidate = Path.cwd()
REPO_ROOT = None
for _ in range(5):
    if (_candidate / '.env').exists() and (_candidate / 'prototypes' / 'lib' / 'rd_parsers.py').exists():
        REPO_ROOT = _candidate
        break
    _candidate = _candidate.parent
if REPO_ROOT is None:
    raise RuntimeError('Could not find REPO_ROOT from cwd')
sys.path.insert(0, str(REPO_ROOT / 'prototypes'))
sys.path.insert(0, str(REPO_ROOT / 'prototypes' / 'lib'))

load_dotenv(REPO_ROOT / '.env')

from lib.rd_parsers import parse_product_page

CLOSET_DATA_DIR = Path(os.environ.get('CLOSET_DATA_DIR', str(Path.home() / '.closet')))
CLOSET_DB = CLOSET_DATA_DIR / 'closet.db'
OUTPUT_DIR = REPO_ROOT / 'prototypes' / 'outputs'
CACHE_DIR = REPO_ROOT / 'prototypes' / 'cache'
URLS_FILE = REPO_ROOT / 'prototypes' / 'inputs' / 'rd-urls2.txt'

CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'DB:       {CLOSET_DB}  (exists: {CLOSET_DB.exists()})')
print(f'Data dir: {CLOSET_DATA_DIR}')
print(f'URLs:     {URLS_FILE}  (exists: {URLS_FILE.exists()})')
print(f'Cache:    {CACHE_DIR}')

/opt/anaconda3/lib/python3.12/site-packages/pandas/core/computation/expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.8.7' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED
/opt/anaconda3/lib/python3.12/site-packages/pandas/core/arrays/masked.py:56: UserWarning: Pandas requires version '1.4.2' or newer of 'bottleneck' (version '1.3.7' currently installed).
  from pandas.core import (


DB:       ~/.closet/closet.db  (exists: True)
Data dir: ~/.closet
URLs:     prototypes/inputs/rd-urls2.txt  (exists: True)
Cache:    prototypes/cache


## 1 — Parse URLs from rd-urls2.txt

Strip comments, blank lines, fragment hashes; deduplicate by base URL.

In [2]:
def parse_urls_file(path: Path) -> list[dict]:
    """Read rd-urls2.txt, return deduplicated list of {url, host, label}."""
    raw = path.read_text()
    urls = []
    current_section = 'unknown'
    for line in raw.split('\n'):
        line = line.strip()
        if not line:
            continue
        if line.startswith('#'):
            current_section = line.lstrip('#').strip()
            continue
        # Normalize: strip fragment, trailing slash
        no_frag = line.split('#')[0]
        parsed = urlparse(no_frag)
        host = parsed.netloc.replace('www.', '').replace('www2.', '')
        normalized = f'{parsed.scheme}://{parsed.netloc}{parsed.path}'
        if parsed.query:
            normalized += f'?{parsed.query}'
        urls.append({'url': line, 'normalized': normalized, 'host': host, 'section': current_section})
    
    # Deduplicate by normalized URL
    seen = set()
    unique = []
    for u in urls:
        if u['normalized'] not in seen:
            seen.add(u['normalized'])
            unique.append(u)
    return unique

urls = parse_urls_file(URLS_FILE)
print(f'Parsed {len(urls)} unique URLs from rd-urls2.txt')
host_counts = Counter(u['host'] for u in urls)
for host, count in host_counts.most_common():
    print(f'  {host}: {count}')

Parsed 31 unique URLs from rd-urls2.txt
  gapcanada.ca: 5
  oldnavy.gapcanada.ca: 5
  hm.com: 3
  asos.com: 3
  etsy.com: 3
  poshmark.ca: 3
  zara.com: 2
  depop.com: 2
  grailed.com: 2
  rothys.com: 1
  kirrinfinch.com: 1
  beefcakeswimwear.com: 1


## 2 — Fetch + Parse Cascade

For each URL: GET the page via httpx (15s timeout), pass HTML through `parse_product_page`.
Results cached to `cache/rd-19-fetches.json` so reruns are free.

In [3]:
FETCH_CACHE = CACHE_DIR / 'rd-19-fetches.json'

if FETCH_CACHE.exists():
    print(f'✓ Loading cached results from {FETCH_CACHE}')
    fetch_results = json.loads(FETCH_CACHE.read_text())
    print(f'  {len(fetch_results)} results loaded')
else:
    fetch_results = []
    print(f'Fetching {len(urls)} URLs with httpx...')
    
    # Browser-like headers to reduce bot-blocking
    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36',
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,image/webp,*/*;q=0.8',
        'Accept-Language': 'en-CA,en;q=0.9,fr;q=0.8',
        'Accept-Encoding': 'gzip, deflate, br',
        'Cache-Control': 'no-cache',
        'DNT': '1',
    }
    
    for i, u in enumerate(urls):
        url = u['url']
        t0 = time.time()
        result = {
            'url': url,
            'host': u['host'],
            'section': u['section'],
            'fetch_ok': False,
            'status_code': None,
            'html_size': 0,
            'fetch_error': None,
        }
        
        try:
            resp = httpx.get(url, headers=headers, timeout=15.0, follow_redirects=True)
            result['status_code'] = resp.status_code
            result['html_size'] = len(resp.text)
            elapsed = time.time() - t0
            
            if resp.status_code == 200:
                result['fetch_ok'] = True
                parsed = parse_product_page(url, resp.text)
                result['parser_used'] = parsed['parser_used']
                result['success'] = parsed['success']
                result['fields'] = parsed['fields']
                if not parsed['success']:
                    result['failure_mode'] = parsed.get('failure_mode', 'unknown')
                print(f'  [{i+1}/{len(urls)}] {u["host"]:25s}  HTTP {resp.status_code}  parser={parsed["parser_used"] or "FAIL"}  {parsed["fields"].get("title", "")[:50] if parsed["fields"].get("title") else "N/A"}  ({elapsed:.1f}s)')
            else:
                result['fetch_ok'] = False
                result['success'] = False
                result['failure_mode'] = f'http_{resp.status_code}'
                result['fields'] = {}
                print(f'  [{i+1}/{len(urls)}] {u["host"]:25s}  HTTP {resp.status_code}  ({elapsed:.1f}s)')
        except Exception as e:
            result['fetch_error'] = str(e)[:200]
            result['success'] = False
            result['failure_mode'] = 'fetch_error'
            result['fields'] = {}
            elapsed = time.time() - t0
            err_msg = str(e)[:80]
            print(f'  [{i+1}/{len(urls)}] {u["host"]:25s}  FETCH ERR: {err_msg}  ({elapsed:.1f}s)')
        
        result['elapsed_s'] = round(elapsed, 1)
        fetch_results.append(result)
    
    FETCH_CACHE.write_text(json.dumps(fetch_results, indent=2))
    print(f'\n✓ Results cached to {FETCH_CACHE}')

✓ Loading cached results from prototypes/cache/rd-19-fetches.json
  31 results loaded


## 3 — Parser Pass/Fail Summary

Per-host success rates, field-level coverage, parser cascade breakdown, failure-mode distribution.

In [4]:
df_p = pd.DataFrame(fetch_results)
total = len(df_p)
success_count = df_p['success'].sum()
fetch_ok_count = df_p['fetch_ok'].sum()

print(f'Total URLs:          {total}')
print(f'HTTP 200 (fetch OK): {fetch_ok_count}/{total} ({fetch_ok_count/total*100:.0f}%)')
print(f'Parser success:      {success_count}/{total} ({success_count/total*100:.0f}%)')
print()

# Per-host
host_stats = df_p.groupby('host').agg(
    total=('success', 'size'),
    successes=('success', 'sum'),
    fetch_ok=('fetch_ok', 'sum'),
).reset_index()
host_stats['rate'] = (host_stats['successes'] / host_stats['total'] * 100).round(0).astype(int)
host_stats = host_stats.sort_values('rate', ascending=False)

print('Per-host success rates:')
print(f'{"Host":35s} {"Total":>5s} {"Fetch":>5s} {"Parsed":>6s} {"Rate":>5s}')
print('-' * 60)
for _, row in host_stats.iterrows():
    print(f'{row["host"]:35s} {row["total"]:5d} {row["fetch_ok"]:5d} {row["successes"]:6d} {row["rate"]:4d}%')
print()

# Parser cascade breakdown
parser_counts = df_p[df_p['success']]['parser_used'].value_counts()
print('Parser cascade breakdown (successful only):')
for parser, count in parser_counts.items():
    print(f'  {parser}: {count} ({count/success_count*100:.0f}%)')

Total URLs:          31
HTTP 200 (fetch OK): 18/31 (58%)
Parser success:      5/31 (16%)

Per-host success rates:
Host                                Total Fetch Parsed  Rate
------------------------------------------------------------
beefcakeswimwear.com                    1     1      1  100%
kirrinfinch.com                         1     1      1  100%
poshmark.ca                             3     3      3  100%
asos.com                                3     0      0    0%
depop.com                               2     0      0    0%
etsy.com                                3     0      0    0%
gapcanada.ca                            5     5      0    0%
grailed.com                             2     0      0    0%
hm.com                                  3     0      0    0%
oldnavy.gapcanada.ca                    5     5      0    0%
rothys.com                              1     1      0    0%
zara.com                                2     2      0    0%

Parser cascade breakdown (succe

In [5]:
# Field-level coverage (% of all URLs, including failures)
fields = ['title', 'image', 'price', 'brand']
print('Field-level coverage (% of all URLs):')
for field in fields:
    count = sum(1 for r in fetch_results if r.get('fields', {}).get(field))
    pct = count / total * 100
    bar = '█' * int(pct / 2) + '░' * (50 - int(pct / 2))
    print(f'  {field:6s}: {count:2d}/{total} ({pct:5.1f}%) {bar}')
print()

# Failure-mode distribution
failures = [r for r in fetch_results if not r['success']]
if failures:
    fail_modes = Counter(r.get('failure_mode', 'unknown') for r in failures)
    print('Failure-mode distribution:')
    for mode, count in fail_modes.most_common():
        print(f'  {mode}: {count}/{len(failures)}')
else:
    print('No failures!')

Field-level coverage (% of all URLs):
  title :  6/31 ( 19.4%) █████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
  image :  6/31 ( 19.4%) █████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
  price :  5/31 ( 16.1%) ████████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
  brand :  4/31 ( 12.9%) ██████░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░

Failure-mode distribution:
  anti_bot: 11/26
  http_403: 10/26
  fetch_error: 3/26
  spa_empty_shell: 2/26


In [6]:
# Detailed per-URL table
print('Per-URL detail:')
print(f'{"Host":25s} {"Parser":>9s} {"Success":>7s} {"Title":50s} {"Price":>8s} {"Brand"}')
print('-' * 130)
for r in fetch_results:
    fld = r.get('fields', {}) or {}
    title = (fld.get('title') or '')[:48]
    price = fld.get('price') or ''
    brand = (fld.get('brand') or '')[:20]
    parser = r.get('parser_used') or 'FAIL'
    status = '✓' if r['success'] else f'✗ {r.get("failure_mode", "unknown")}'
    print(f'{r["host"]:25s} {parser:>9s} {status:>7s} {title:50s} {str(price):>8s} {brand}')

Per-URL detail:
Host                         Parser Success Title                                                 Price Brand
----------------------------------------------------------------------------------------------------------------------------------
hm.com                         FAIL ✗ http_403                                                             
hm.com                         FAIL ✗ http_403                                                             
hm.com                         FAIL ✗ http_403                                                             
gapcanada.ca                   FAIL ✗ anti_bot                                                             
gapcanada.ca                   FAIL ✗ anti_bot                                                             
gapcanada.ca                   FAIL ✗ anti_bot                                                             
gapcanada.ca                   FAIL ✗ anti_bot                                                 

## 4 — Load Closet Items from Dev DB

In [7]:
def fetch_closet_items(db_path: Path) -> list[dict]:
    conn = sqlite3.connect(f'file:{db_path}?mode=ro', uri=True)
    conn.row_factory = sqlite3.Row
    rows = conn.execute('''
        SELECT id, category, subcategory, colors, brand, material, season,
               occasion, tags, ai_description, wear_context, image_uri, primary_color
        FROM items
        ORDER BY id
    ''').fetchall()
    conn.close()
    
    items = []
    for r in rows:
        d = dict(r)
        for jf in ('colors', 'season', 'occasion', 'tags'):
            try:
                d[jf] = json.loads(d[jf]) if d[jf] else []
            except (json.JSONDecodeError, TypeError):
                d[jf] = []
        # Resolve image path
        if d.get('image_uri'):
            p = CLOSET_DATA_DIR / d['image_uri']
            d['abs_image_path'] = str(p) if p.exists() else None
        else:
            d['abs_image_path'] = None
        items.append(d)
    return items

closet_items = fetch_closet_items(CLOSET_DB)
items_with_img = [i for i in closet_items if i['abs_image_path']]

print(f'Closet items in DB:        {len(closet_items)}')
print(f'With image on disk:        {len(items_with_img)}')
print(f'Categories: {dict(Counter(i["category"] for i in closet_items).most_common())}')
print(f'Brands:     {dict(Counter((i["brand"] or "unknown") for i in closet_items).most_common(8))}')

Closet items in DB:        71
With image on disk:        35
Categories: {'tops': 23, 'bottoms': 14, 'accessories': 12, 'outerwear': 11, 'shoes': 8, 'Shirt': 2, '🧕': 1}
Brands:     {'unknown': 69, 'adidas': 1, 'fes': 1}


## 5 — Build Text Descriptions + TF-IDF Vectors

For each scraped product and each closet item, construct a text blob from all available metadata. Then vectorize with TF-IDF — this is our "hash-stub-embed" (deterministic, no API creds needed).

In [8]:
def scraped_blob(product: dict) -> str:
    """Build text description from parsed product fields."""
    f = product.get('fields', {}) or {}
    parts = []
    if f.get('title'):
        parts.append(f['title'])
    if f.get('brand'):
        parts.append(f"brand:{f['brand']}")
    if f.get('price'):
        parts.append(f"price:{f['price']}")
    # Include host as implicit brand/category signal
    parts.append(f"store:{product['host']}")
    return ' '.join(parts)

def closet_blob(item: dict) -> str:
    """Build text description from closet item metadata."""
    parts = [
        item.get('category', ''),
        item.get('subcategory', '') or '',
        ' '.join(item.get('colors') or []),
        f"brand:{item.get('brand')}" if item.get('brand') else '',
        item.get('material', '') or '',
        ' '.join(item.get('season') or []),
        ' '.join(item.get('occasion') or []),
        ' '.join(item.get('tags') or []),
        item.get('ai_description', '') or '',
        item.get('wear_context', '') or '',
        item.get('primary_color', '') or '',
    ]
    return ' '.join(p for p in parts if p) or item['category']

# Build blobs for successful parses
successes = [r for r in fetch_results if r['success']]
scraped_blobs = [scraped_blob(r) for r in successes]

# Build blobs for all closet items (including those without images)
all_closet_blobs = [closet_blob(i) for i in closet_items]

print(f'Scraped products to correlate: {len(scraped_blobs)}')
print(f'Closet items to match against: {len(all_closet_blobs)}')
print()
print('Sample scraped blobs:')
for b in scraped_blobs[:3]:
    print(f'  {b[:120]}...')
print()
print('Sample closet blobs:')
for b in all_closet_blobs[:3]:
    print(f'  {b[:120]}...')

Scraped products to correlate: 5
Closet items to match against: 71

Sample scraped blobs:
  Androgynous Linen Camp Shirt | Kirrin Finch price:115 store:kirrinfinch.com...
  The Charmer - LONG brand:Beefcake Swimwear price:155.0 store:beefcakeswimwear.com...
  90’s Montreal Canadiens NHL Ilanco Rugby Polo Shirt Deadstock Size: S brand:Ilanco price:65.0 store:poshmark.ca...

Sample closet blobs:
  accessories beanie grey dark grey knitted fabric fall casual everyday wear winter wear A ribbed grey knitted beanie feat...
  tops button-down shirt white blue cotton spring casual classic everyday workwear A white button-down shirt featuring thi...
  tops button-down shirt navy blue cotton blend spring casual everyday wear classic timeless A navy blue long-sleeved butt...


In [9]:
# TF-IDF vectorization
vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    max_features=500,
    ngram_range=(1, 2),
    sublinear_tf=True,
    analyzer='word',
)

all_docs = scraped_blobs + all_closet_blobs
tfidf_matrix = vectorizer.fit_transform(all_docs)

scraped_vecs = tfidf_matrix[:len(scraped_blobs)]
closet_vecs = tfidf_matrix[len(scraped_blobs):]

print(f'TF-IDF vocabulary size: {len(vectorizer.vocabulary_)}')
print(f'Scraped vectors shape:  {scraped_vecs.shape}')
print(f'Closet vectors shape:   {closet_vecs.shape}')

# Cosine similarity: each scraped product vs all closet items
cos_sims = cosine_similarity(scraped_vecs, closet_vecs)

TF-IDF vocabulary size: 500
Scraped vectors shape:  (5, 500)
Closet vectors shape:   (71, 500)


## 6 — Top-5 Closet Matches Per Scraped Product

For each successfully parsed product, show the top-5 closet items by TF-IDF cosine similarity, with thumbnails.

In [10]:
def get_top_k(row_idx: int, k: int = 5) -> list[dict]:
    """Return top-k closet matches for a given scraped-product row."""
    scores = cos_sims[row_idx]
    top_indices = np.argsort(-scores)[:k]
    results = []
    for j in top_indices:
        results.append({
            'item': closet_items[j],
            'score': float(scores[j]),
            'rank': len(results) + 1,
        })
    return results

# Show one example
ex = successes[0]
matches = get_top_k(0)
print(f'Scraped: {ex["fields"].get("title", "N/A")[:60]}')
print()
for m in matches:
    it = m['item']
    print(f'  [{m["rank"]}] score={m["score"]:.3f}  {it["category"]}/{it.get("subcategory", "")}  '
          f'{(it.get("colors") or ["?"])[0]}  {it.get("brand") or "unbranded"}')

Scraped: Androgynous Linen Camp Shirt | Kirrin Finch

  [1] score=0.227  Shirt/None  ?  unbranded
  [2] score=0.052  tops/t-shirt  black  unbranded
  [3] score=0.051  tops/shirt  black  unbranded
  [4] score=0.047  tops/t-shirt  yellow  unbranded
  [5] score=0.043  tops/button-down shirt  red  unbranded


In [11]:
# Render all scraped products with their top-5 closet matches
k = 5
n_products = len(successes)
cols = 1 + k  # product image + top-k matches

fig, axes = plt.subplots(n_products, cols, figsize=(3 * cols, 3.2 * n_products))
if n_products == 1:
    axes = np.array([axes])

for row_idx, (product, ax_row) in enumerate(zip(successes, axes)):
    fld = product.get('fields', {}) or {}
    title = (fld.get('title') or 'Untitled')[:35]
    
    # Product column: show text card (product image URL is remote — skip for speed)
    ax_product = ax_row[0]
    ax_product.text(0.5, 0.5, f'{title}\n{fld.get("brand", "") or ""}\n{fld.get("price", "") or ""} {fld.get("currency", "")}',
                    ha='center', va='center', fontsize=7, transform=ax_product.transAxes,
                    bbox=dict(boxstyle='round', facecolor='#e8f5e9', alpha=0.9))
    ax_product.set_xlim(0, 1); ax_product.set_ylim(0, 1)
    ax_product.axis('off')
    ax_product.set_title(f'{product["host"]}\n{product["parser_used"]}', fontsize=7, pad=2)
    
    # Top-K matches
    matches = get_top_k(row_idx, k)
    for col_idx, m in enumerate(matches):
        ax = ax_row[col_idx + 1]
        it = m['item']
        img_path = it.get('abs_image_path')
        if img_path:
            try:
                img = Image.open(img_path)
                ax.imshow(img)
            except Exception:
                ax.text(0.5, 0.5, 'NO IMG', ha='center', va='center', fontsize=8, transform=ax.transAxes)
        else:
            ax.text(0.5, 0.5, 'NO IMG', ha='center', va='center', fontsize=8, transform=ax.transAxes)
        
        color = (it.get('colors') or ['?'])[0]
        cat = it['category']
        brand = (it.get('brand') or '')[:12]
        ax.set_title(f'#{m["rank"]} cos={m["score"]:.2f}\n{cat} · {color}', fontsize=7)
        ax.axis('off')

plt.suptitle(f'Top-{k} Closet Matches per Scraped Product (TF-IDF cosine)', fontsize=13, y=0.995)
plt.tight_layout()
out_png = OUTPUT_DIR / '19-correlation-topk.png'
plt.savefig(out_png, dpi=150, bbox_inches='tight')
print(f'Saved grid to {out_png}')
plt.show()

Saved grid to prototypes/outputs/19-correlation-topk.png


[image removed from the public copy: third-party or personal photos]


## 7 — Summary Stats

Correlation metrics: mean top-1 cosine, mean top-5 cosine, category-overlap rate.

In [12]:
all_top1_scores = []
all_top5_mean_scores = []
all_topk = []

print(f'{"Host":25s} {"Product":35s} {"Top-1 Cos":>9s} {"Top-5 μ":>8s}')
print('-' * 85)

for row_idx, product in enumerate(successes):
    top5 = get_top_k(row_idx, 5)
    top1_score = top5[0]['score'] if top5 else 0
    top5_mean = np.mean([m['score'] for m in top5])
    all_top1_scores.append(top1_score)
    all_top5_mean_scores.append(top5_mean)
    all_topk.append(top5)
    
    title = (product['fields'].get('title') or 'N/A')[:33]
    print(f'{product["host"]:25s} {title:35s} {top1_score:9.3f} {top5_mean:8.3f}')

print()
print(f'Mean top-1 cosine: {np.mean(all_top1_scores):.3f}')
print(f'Mean top-5 μ cosine: {np.mean(all_top5_mean_scores):.3f}')
print(f'Max top-1 cosine: {np.max(all_top1_scores):.3f}')
print(f'Min top-1 cosine: {np.min(all_top1_scores):.3f}')

Host                      Product                             Top-1 Cos  Top-5 μ
-------------------------------------------------------------------------------------
kirrinfinch.com           Androgynous Linen Camp Shirt | Ki       0.227    0.084
beefcakeswimwear.com      The Charmer - LONG                      0.044    0.042
poshmark.ca               90’s Montreal Canadiens NHL Ilanc       0.131    0.054
poshmark.ca               Travis Mathew men’s XL golf wind        0.058    0.044
poshmark.ca               Wrangler Short Sleeve Button Down       0.107    0.067

Mean top-1 cosine: 0.113
Mean top-5 μ cosine: 0.058
Max top-1 cosine: 0.227
Min top-1 cosine: 0.044


## 8 — Image-Embedding Correlation (gemini-embedding-2-preview)

**The real test.** All 5 successfully parsed products have a product image URL in `fields.image`. We download each, embed it with `gemini-embedding-2-preview` (1408-d multimodal — the same model the app uses for `variant=image` in the `embeddings` table), then cosine against the 37 pre-computed closet-item image embeddings. No text, no tags, no brand — pure visual signal.

If image-embedding top-1 cosine jumps to 0.5+ where TF-IDF topped out at 0.227, the matching idea is sound and the only remaining work is unblocking the parser.



In [13]:
# Download product images from parsed URLs
import io, base64, hashlib
from PIL import Image

# Browser-like headers for image CDN requests
headers = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/131.0.0.0 Safari/537.36',
    'Accept': 'image/webp,image/*,*/*;q=0.8',
    'Accept-Language': 'en-CA,en;q=0.9',
    'DNT': '1',
}

PRODUCT_IMG_DIR = CACHE_DIR / 'rd-19-product-images'
PRODUCT_IMG_DIR.mkdir(parents=True, exist_ok=True)

def download_product_image(product, timeout=15.0):
    img_url = (product.get('fields') or {}).get('image')
    if not img_url:
        return None
    slug = product['host'].replace('.', '-') + '_' + hashlib.md5(product['url'].encode()).hexdigest()[:8]
    out_path = PRODUCT_IMG_DIR / f'{slug}.jpg'
    if out_path.exists():
        return str(out_path)
    try:
        resp = httpx.get(img_url, headers=headers, timeout=timeout, follow_redirects=True)
        if resp.status_code == 200 and len(resp.content) > 1024:
            img = Image.open(io.BytesIO(resp.content)).convert('RGB')
            w, h = img.size
            if max(w, h) > 1200:
                ratio = 1200 / max(w, h)
                img = img.resize((int(w*ratio), int(h*ratio)), Image.LANCZOS)
            img.save(out_path, 'JPEG', quality=80)
            return str(out_path)
    except Exception as e:
        print(f'  download failed: {img_url[:80]} - {e}')
    return None

print(f'Downloading product images for {len(successes)} parsed products...')
for i, p in enumerate(successes):
    fld = p.get('fields', {}) or {}
    local_path = download_product_image(p)
    p['product_img_path'] = local_path
    status = 'OK' if local_path else 'FAIL'
    title = (fld.get('title') or '')[:40]
    print(f'  [{i+1}/5] {status} {p["host"]:25s} {title:40s}')



  [1/5] OK kirrinfinch.com           Androgynous Linen Camp Shirt | Kirrin Fi
  [2/5] OK beefcakeswimwear.com      The Charmer - LONG                      
  [3/5] OK poshmark.ca               90’s Montreal Canadiens NHL Ilanco Rugby
  [4/5] OK poshmark.ca               Travis Mathew men’s XL golf wind rain ja
  [5/5] OK poshmark.ca               Wrangler Short Sleeve Button Down Plaid 


In [14]:
# Check gemini-embedding-2-preview availability
import io, base64
from PIL import Image
from google import genai

gemini_client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
EMBED_MODEL = 'gemini-embedding-2-preview'

print(f'Embedding model: {EMBED_MODEL}')
try:
    test_img = Image.new('RGB', (100, 100), color='blue')
    buf = io.BytesIO()
    test_img.save(buf, format='JPEG', quality=80)
    b64 = base64.b64encode(buf.getvalue()).decode()
    result = gemini_client.models.embed_content(
        model=EMBED_MODEL,
        contents=[{'role': 'user', 'parts': [{'inline_data': {'mime_type': 'image/jpeg', 'data': b64}}]}],
        config={'outputDimensionality': 1408},
    )
    if hasattr(result, 'embeddings') and result.embeddings and result.embeddings[0]:
        dim = len(result.embeddings[0].values)
        print(f'  OK embed API works - returned {dim}-dim vector')
        EMBED_API_WORKS = True
    else:
        print(f'  embed response: {type(result)}')
        print(f'  has embedding attr: {hasattr(result, "embedding")}')
        if hasattr(result, "text"):
            print(f'  result.text: {result.text[:200]}')
        EMBED_API_WORKS = False
except Exception as e:
    print(f'  embed API failed: {e}')
    import traceback
    traceback.print_exc()
    EMBED_API_WORKS = False

print(f'\nEMBED_API_WORKS = {EMBED_API_WORKS}')





Embedding model: gemini-embedding-2-preview


  OK embed API works - returned 1408-dim vector

EMBED_API_WORKS = True


In [15]:
# Embed all product images via gemini-embedding-2-preview
import io, base64
from PIL import Image
from google import genai

if not EMBED_API_WORKS:
    print('Embedding API not available - skipping image-embedding correlation')
    df_product_embeds = pd.DataFrame()
else:
    gemini_client = genai.Client(api_key=os.environ['GEMINI_API_KEY'])
    EMBED_MODEL = 'gemini-embedding-2-preview'

    EMBED_CACHE = CACHE_DIR / 'rd-19-product-image-embeds.parquet'

    if EMBED_CACHE.exists():
        print(f'Loading cached product embeds from {EMBED_CACHE}')
        df_product_embeds = pd.read_parquet(EMBED_CACHE)
    else:
        rows = []
        for i, p in enumerate(successes):
            path = p.get('product_img_path')
            title = (p.get('fields') or {}).get('title', 'N/A')[:50]
            if path:
                t0 = time.time()
                try:
                    img = Image.open(path).convert('RGB')
                    buf = io.BytesIO()
                    img.save(buf, format='JPEG', quality=80)
                    b64 = base64.b64encode(buf.getvalue()).decode()
                    result = gemini_client.models.embed_content(
                        model=EMBED_MODEL,
                        contents=[{'role': 'user', 'parts': [{'inline_data': {'mime_type': 'image/jpeg', 'data': b64}}]}],
        config={'outputDimensionality': 1408},
                    )
                    if hasattr(result, 'embeddings') and result.embeddings and result.embeddings[0]:
                        vec = np.array(result.embeddings[0].values, dtype=np.float32)
                        vec = vec / (np.linalg.norm(vec) + 1e-12)
                        dt = time.time() - t0
                        rows.append({'idx': i, 'host': p['host'], 'title': title,
                                     'vec': vec.tolist(), 'elapsed_s': round(dt, 1)})
                        print(f'  [{i+1}/5] OK {p["host"]:25s} {title:50s}  dim={len(vec)}  ({dt:.1f}s)')
                    else:
                        print(f'  [{i+1}/5] SKIP {p["host"]:25s} {title:50s}  no embedding in response')
                except Exception as e:
                    print(f'  [{i+1}/5] ERR {p["host"]:25s} {title:50s}  {e}')
            else:
                print(f'  [{i+1}/5] NOP {p["host"]:25s} {title:50s}  no product image')
        df_product_embeds = pd.DataFrame(rows)
        df_product_embeds.to_parquet(EMBED_CACHE)
        print(f'\nCached to {EMBED_CACHE}')

    print(f'\nProduct images embedded: {len(df_product_embeds)}')





  [1/5] OK kirrinfinch.com           Androgynous Linen Camp Shirt | Kirrin Finch         dim=1408  (1.0s)


  [2/5] OK beefcakeswimwear.com      The Charmer - LONG                                  dim=1408  (0.8s)


  [3/5] OK poshmark.ca               90’s Montreal Canadiens NHL Ilanco Rugby Polo Shir  dim=1408  (0.7s)


  [4/5] OK poshmark.ca               Travis Mathew men’s XL golf wind rain jacket        dim=1408  (0.7s)


  [5/5] OK poshmark.ca               Wrangler Short Sleeve Button Down Plaid Shirt XL M  dim=1408  (0.8s)

Cached to prototypes/cache/rd-19-product-image-embeds.parquet

Product images embedded: 5


In [16]:
# Load closet item image embeddings from DB
def load_closet_image_embeddings():
    conn = sqlite3.connect(f'file:{CLOSET_DB}?mode=ro', uri=True)
    rows = conn.execute("""
        SELECT e.ref_id, e.vector, e.model_version,
               i.category, i.subcategory, i.colors, i.brand, i.material,
               i.ai_description, i.image_uri, i.primary_color
        FROM embeddings e
        JOIN items i ON i.id = e.ref_id
        WHERE e.variant = 'image' AND e.user_id = 'self'
    """).fetchall()
    conn.close()
    
    result = {}
    for r in rows:
        ref_id, vec_bytes, model_ver, cat, subcat, colors_json, brand, material, ai_desc, img_uri, pcolor = r
        vec = np.frombuffer(vec_bytes, dtype=np.float32).copy()
        vec = vec / (np.linalg.norm(vec) + 1e-12)
        colors = json.loads(colors_json) if colors_json else []
        img_path = CLOSET_DATA_DIR / img_uri if img_uri else None
        result[ref_id] = (vec, {
            'id': ref_id, 'category': cat, 'subcategory': subcat, 'colors': colors,
            'brand': brand, 'material': material, 'aiDescription': ai_desc,
            'primaryColor': pcolor, 'imageUri': img_uri,
            'abs_image_path': str(img_path) if img_path and img_path.exists() else None,
        })
    return result

closet_img_embeds = load_closet_image_embeddings()
closet_ids = list(closet_img_embeds.keys())

if closet_ids:
    D = len(closet_img_embeds[closet_ids[0]][0])
    closet_img_matrix = np.zeros((len(closet_ids), D), dtype=np.float32)
    for i, cid in enumerate(closet_ids):
        closet_img_matrix[i] = closet_img_embeds[cid][0]
    print(f'Closet image embeddings: {len(closet_ids)} items, {D}-dim')
    cats = Counter(closet_img_embeds[cid][1]['category'] for cid in closet_ids)
    print(f'Categories: {dict(cats.most_common())}')
else:
    closet_img_matrix = np.zeros((0, 1), dtype=np.float32)
    print('No closet image embeddings found!')



Closet image embeddings: 37 items, 1408-dim
Categories: {'tops': 14, 'bottoms': 8, 'outerwear': 6, 'accessories': 4, 'shoes': 3, 'Shirt': 1, '🧕': 1}


In [17]:
# Image-embedding top-5 per product + visual grid
if len(df_product_embeds) == 0 or len(closet_ids) == 0:
    print('Not enough data for image-embedding correlation')
else:
    product_vecs = np.array([np.array(v, dtype=np.float32) for v in df_product_embeds['vec'].values])
    img_cos_sims = cosine_similarity(product_vecs, closet_img_matrix)
    
    n_products = len(product_vecs)
    k = 5
    cols = 1 + k
    fig, axes = plt.subplots(n_products, cols, figsize=(3 * cols, 3.2 * n_products))
    if n_products == 1:
        axes = np.array([axes])
    
    img_top1 = []
    img_top5_means = []
    
    print(f'{"Host":25s} {"Product":35s} {"Top-1 Cos":>9s} {"Top-5 mu":>8s}')
    print('-' * 85)
    
    for row_idx, (ax_row, prod_row) in enumerate(zip(axes, df_product_embeds.iterrows())):
        _, prod = prod_row
        scores = img_cos_sims[row_idx]
        top_indices = np.argsort(-scores)[:k]
        top1 = float(scores[top_indices[0]])
        top5_mean = float(np.mean(scores[top_indices]))
        img_top1.append(top1)
        img_top5_means.append(top5_mean)
        
        print(f'{prod["host"]:25s} {prod["title"]:35s} {top1:9.3f} {top5_mean:8.3f}')
        
        ax_product = ax_row[0]
        prod_img_path = successes[prod['idx']].get('product_img_path')
        if prod_img_path:
            ax_product.imshow(Image.open(prod_img_path))
        else:
            ax_product.text(0.5, 0.5, 'NO IMG', ha='center', va='center', fontsize=8, transform=ax_product.transAxes)
        ax_product.set_title(f'{prod["host"]}\n{prod["title"][:30]}', fontsize=7, pad=2)
        ax_product.axis('off')
        
        for col_idx, j in enumerate(top_indices):
            ax = ax_row[col_idx + 1]
            cid = closet_ids[j]
            _, citem = closet_img_embeds[cid]
            img_path = citem.get('abs_image_path')
            if img_path:
                ax.imshow(Image.open(img_path))
            else:
                ax.text(0.5, 0.5, 'NO IMG', ha='center', va='center', fontsize=8, transform=ax.transAxes)
            color = (citem.get('colors') or ['?'])[0]
            score = float(scores[j])
            ax.set_title(f'#{col_idx+1} cos={score:.2f}\n{citem["category"]} * {color}', fontsize=7)
            ax.axis('off')
    
    print()
    print(f'IMAGE-EMBED Mean top-1 cosine: {np.mean(img_top1):.3f}')
    print(f'IMAGE-EMBED Mean top-5 mu cosine: {np.mean(img_top5_means):.3f}')
    print(f'IMAGE-EMBED Max top-1 cosine:   {np.max(img_top1):.3f}')
    print(f'IMAGE-EMBED Min top-1 cosine:   {np.min(img_top1):.3f}')
    
    plt.suptitle(f'Image-Embed Top-{k} Closet Matches (gemini-embedding-2-preview, 1408-d)', fontsize=13, y=0.995)
    plt.tight_layout()
    out_png = OUTPUT_DIR / '19-correlation-image-embed-topk.png'
    plt.savefig(out_png, dpi=150, bbox_inches='tight')
    print(f'\nSaved grid to {out_png}')
    plt.show()



Host                      Product                             Top-1 Cos Top-5 mu
-------------------------------------------------------------------------------------
kirrinfinch.com           Androgynous Linen Camp Shirt | Kirrin Finch     0.660    0.650


beefcakeswimwear.com      The Charmer - LONG                      0.709    0.683
poshmark.ca               90’s Montreal Canadiens NHL Ilanco Rugby Polo Shir     0.698    0.686
poshmark.ca               Travis Mathew men’s XL golf wind rain jacket     0.700    0.677
poshmark.ca               Wrangler Short Sleeve Button Down Plaid Shirt XL M     0.681    0.674

IMAGE-EMBED Mean top-1 cosine: 0.690
IMAGE-EMBED Mean top-5 mu cosine: 0.674
IMAGE-EMBED Max top-1 cosine:   0.709
IMAGE-EMBED Min top-1 cosine:   0.660



Saved grid to prototypes/outputs/19-correlation-image-embed-topk.png


[image removed from the public copy: third-party or personal photos]


## 9 — TF-IDF vs Image-Embedding: The Acid Test

Compare top-1 cosine scores across both methods for the exact same 5 products.



In [18]:
# Side-by-side comparison
print('TF-IDF (text) vs Image-Embedding (visual) - top-1 cosine per product:')
print(f'{"Host":25s} {"Product":30s} {"TF-IDF":>7s} {"Img-Embed":>10s} {"Delta":>7s}')
print('-' * 85)

tfidf_top1_by_idx = {}
for row_idx, product in enumerate(successes):
    top5 = get_top_k(row_idx, 5)
    tfidf_top1_by_idx[row_idx] = top5[0]['score'] if top5 else 0

img_top1_by_idx = {}
if len(df_product_embeds) > 0 and len(closet_ids) > 0:
    product_vecs = np.array([np.array(v, dtype=np.float32) for v in df_product_embeds['vec'].values])
    img_cos_sims = cosine_similarity(product_vecs, closet_img_matrix)
    for row_idx, prod_row in enumerate(df_product_embeds.iterrows()):
        _, prod = prod_row
        img_top1_by_idx[prod['idx']] = float(np.max(img_cos_sims[row_idx]))

for row_idx, product in enumerate(successes):
    title = (product.get('fields') or {}).get('title', 'N/A')[:28]
    tfidf_score = tfidf_top1_by_idx.get(row_idx, 0)
    img_score = img_top1_by_idx.get(row_idx, None)
    if img_score is not None:
        delta = img_score - tfidf_score
        if delta > 0.05:
            direction = 'IMG wins'
        elif delta < -0.05:
            direction = 'TEXT wins'
        else:
            direction = 'tie'
        print(f'{product["host"]:25s} {title:30s} {tfidf_score:7.3f} {img_score:10.3f} {delta:+7.3f}  {direction}')
    else:
        print(f'{product["host"]:25s} {title:30s} {tfidf_score:7.3f} {"N/A":>10s}  (no img embed)')

print()
img_mean = np.mean(list(img_top1_by_idx.values())) if img_top1_by_idx else 0
tfidf_mean = np.mean(list(tfidf_top1_by_idx.values())) if tfidf_top1_by_idx else 0
print(f'Mean TF-IDF top-1:         {tfidf_mean:.3f}')
print(f'Mean Image-Embed top-1:    {img_mean:.3f}')
print(f'Gap:                        {img_mean - tfidf_mean:+.3f}')
print()
if img_mean > 0.4:
    print('Image embedding consistently beats TF-IDF - visual signal exists.')
    print('  The parser unblock is the only remaining work.')
elif img_mean > 0.2:
    print('Image embedding shows some signal above random - matching idea has legs but small sample (n=5).')
else:
    print('Image-embedding top-1 still hovers near TF-IDF levels.')
    print('  Either n=5 products are too visually distant from the closet, or')
    print('  embedding-2-preview lacks the fashion-domain precision of Vertex multimodalembedding@001.')



TF-IDF (text) vs Image-Embedding (visual) - top-1 cosine per product:
Host                      Product                         TF-IDF  Img-Embed   Delta
-------------------------------------------------------------------------------------
kirrinfinch.com           Androgynous Linen Camp Shirt     0.227      0.660  +0.433  IMG wins
beefcakeswimwear.com      The Charmer - LONG               0.044      0.709  +0.665  IMG wins
poshmark.ca               90’s Montreal Canadiens NHL      0.131      0.698  +0.567  IMG wins
poshmark.ca               Travis Mathew men’s XL golf      0.058      0.700  +0.642  IMG wins
poshmark.ca               Wrangler Short Sleeve Button     0.107      0.681  +0.574  IMG wins

Mean TF-IDF top-1:         0.113
Mean Image-Embed top-1:    0.690
Gap:                        +0.576

Image embedding consistently beats TF-IDF - visual signal exists.
  The parser unblock is the only remaining work.


## 10 — Findings

**What this notebook validates:**
1. Which parser strategies succeed/fail per e-commerce site — real HTTP, not mock data.
2. How well scraped TF-IDF text vectors match against closet items.
3. Whether **image-embedding** cosine (gemini-embedding-2-preview) beats TF-IDF — the critical acid test for the fit-score pipeline.
4. Where the three-parser cascade breaks — SPA shells, anti-bot walls, schema-gap sites.

**Limitations:**
- Vertex `multimodalembedding@001` (1408-d fashion-tuned) is not wired — the Gemini `embedding-2-preview` proxy may be weaker.
- n=5 products is small. Parser unblock (headless browser or retailer API access) is needed to get a meaningful sample.
- Closet has only 37 items with image embeddings — fine for dev, sparse for production.

**Promotion path:** Open a feature worktree for `shopping-integration` Phase 1 (save shopping URL → parse → embed → show closet visual matches).

